In [ ]:
# ConQur for Fibrinogen

In [ ]:
# Load phyloseq object
ps <- readRDS("/scratch/negishi/ldas/COW_INFLAMMATION/DADA2_inputs/GP_ps.st.decontam.cleaned.nomock.rds")

# Extract OTU/ASV table as a matrix
otu_mat <- as(otu_table(ps), "matrix")

# Check orientation: if taxa are in rows, transpose it
if (taxa_are_rows(ps)) {
  otu_mat <- t(otu_mat)
}

# Confirm dimensions: rows = samples, columns = taxa
dim(otu_mat)
head(otu_mat[, 1:6])  # View first 6 taxa



In [ ]:
# 2. Extract and format metadata
# -------------------------
meta <- data.frame(sample_data(ps))
meta$ExtractionDate <- as.Date(meta$ExtractionDate, format = "%m/%d/%Y")
meta$ExtractionDate_cat <- as.factor(meta$ExtractionDate)
meta$Parity_numeric <- as.numeric(as.character(meta$Parity))
meta$Parity_Group <- factor(ifelse(meta$Parity_numeric <= 3, "Group1", "Group2"))
meta$BCS <- as.numeric(as.character(meta$BCS))
meta$BCS_Group <- factor(cut(meta$BCS,
                             breaks = c(-Inf, 2.5, 3.5, Inf),
                             labels = c("Low", "Medium", "High"),
                             right = TRUE))
meta$Fibrinogen_InflamStat <- factor(meta$Fibrinogen_InflamStat, levels = c("Normal", "Elevated"))
meta$Haptoglobin_InflamStat <- factor(meta$Haptoglobin_InflamStat, levels = c("Normal", "Elevated"))

In [ ]:
# Define Batch id
batchid <- meta$ExtractionDate_cat  # batch variable (factor)

In [ ]:
# Define Covariates
covariates_Fibrinogen <- meta[, c("Fibrinogen_InflamStat", "Parity_Group", "BCS_Group", "DIM_Collected")]

In [ ]:
# Check is.na 
colSums(is.na(covariates_Fibrinogen))
stopifnot(rownames(otu_mat) == rownames(covariates_Fibrinogen))
stopifnot(rownames(otu_mat) == rownames(meta))

In [ ]:
# Define Batch refference
batch_ref <- names(sort(table(batchid), decreasing = TRUE))[1]  # the most common batch

In [ ]:
library(ConQuR)
library(foreach)

res_conqur <- ConQuR(
  tax_tab = otu_mat,
  batchid = batchid,
  covariates = covariates_Fibrinogen,
  batch_ref = batch_ref,
  logistic_lasso = FALSE,
  quantile_type = "standard",
  simple_match = FALSE,
  lambda_quantile = "2p/n",
  interplt = FALSE,
  delta = 0.4999,
  taus = seq(0.005, 0.995, by = 0.005),
  num_core = 2
)


In [ ]:
saveRDS(res_conqur, "ConQuR_output_Fibrinogen.rds")

In [ ]:
library(phyloseq)
library(vegan)
library(compositions) # for clr()

otu_raw <- as(otu_table(ps), "matrix")
if (!taxa_are_rows(ps)) otu_raw <- t(otu_raw)

otu_corrected <- t(res_conqur)  # ASVs x samples

# Keep only taxa with variance > 0
otu_raw <- otu_raw[apply(otu_raw, 1, var) > 0, ]
otu_corrected <- otu_corrected[apply(otu_corrected, 1, var) > 0, ]

# Match metadata order to samples
meta_df <- meta[colnames(otu_raw), ]

# Factors
meta_df$batch <- factor(meta_df$ExtractionDate_cat)
meta_df$group <- factor(meta_df$Fibrinogen_InflamStat, levels = c("Normal", "Elevated"))

# -------------------------
# 6. Distances
# -------------------------
## Bray–Curtis
bray_raw <- vegdist(t(otu_raw / rowSums(otu_raw)), method = "bray")
bray_corrected <- vegdist(t(otu_corrected / rowSums(otu_corrected)), method = "bray")

## Aitchison
clr_raw <- t(clr(otu_raw + 1e-6))
clr_corrected <- t(clr(otu_corrected + 1e-6))
aitch_raw <- dist(clr_raw, method = "euclidean")
aitch_corrected <- dist(clr_corrected, method = "euclidean")

# -------------------------
# 7. PERMANOVA before & after correction
# -------------------------
cat("\n--- PERMANOVA: Bray (pre) ---\n")
print(adonis2(bray_raw ~ batch + group, data = meta_df, permutations = 999, by = "margin"))

cat("\n--- PERMANOVA: Bray (post) ---\n")
print(adonis2(bray_corrected ~ batch + group, data = meta_df, permutations = 999, by = "margin"))

cat("\n--- PERMANOVA: Aitchison (pre) ---\n")
print(adonis2(aitch_raw ~ batch + group, data = meta_df, permutations = 999, by = "margin"))

cat("\n--- PERMANOVA: Aitchison (post) ---\n")
print(adonis2(aitch_corrected ~ batch + group, data = meta_df, permutations = 999, by = "margin"))